# Diabetes Data Cleaning Check

This notebook inspects `diabetes.csv` for data quality issues and builds a cleaned dataset ready for modeling.

Checks included:
- Shape, schema, missing values, duplicates
- Zero-as-missing values in medical columns
- Outlier detection (IQR method)
- Class balance

Cleaning actions included:
- Replace impossible zeros with `NaN` in key medical columns
- Impute with median values
- Optional outlier clipping using IQR bounds

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

In [2]:
df = pd.read_csv('diabetes.csv')
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
df.head()

Shape: (768, 9)

Dtypes:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
print('Duplicate rows:', int(df.duplicated().sum()))
print('\nMissing values by column:')
print(df.isna().sum())

Duplicate rows: 0

Missing values by column:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
medical_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_report = pd.DataFrame({
    'zero_count': [(df[c] == 0).sum() for c in medical_zero_cols],
    'zero_percent': [((df[c] == 0).mean() * 100) for c in medical_zero_cols],
}, index=medical_zero_cols).sort_values('zero_count', ascending=False)

zero_report

,zero_count,zero_percent
Insulin,374,48.697917
SkinThickness,227,29.557292
BloodPressure,35,4.557292
BMI,11,1.432292
Glucose,5,0.651042


In [5]:
def iqr_outlier_report(dataframe):
    rows = []
    for col in dataframe.columns:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n = int(((dataframe[col] < lower) | (dataframe[col] > upper)).sum())
        rows.append({
            'column': col,
            'outlier_count': n,
            'outlier_percent': round((n / len(dataframe)) * 100, 2),
            'lower_bound': round(lower, 3),
            'upper_bound': round(upper, 3),
        })
    return pd.DataFrame(rows).sort_values('outlier_count', ascending=False)

iqr_report = iqr_outlier_report(df)
iqr_report

,column,outlier_count,outlier_percent,lower_bound,upper_bound
2,BloodPressure,45,5.86,35.000,107.000
4,Insulin,34,4.43,-190.875,318.125
6,DiabetesPedigreeFunction,29,3.78,-0.330,1.200
5,BMI,19,2.47,13.350,50.550
7,Age,9,1.17,-1.500,66.500
1,Glucose,5,0.65,37.125,202.125
0,Pregnancies,4,0.52,-6.500,13.500
3,SkinThickness,1,0.13,-48.000,80.000
8,Outcome,0,0.00,-1.500,2.500


In [6]:
print('Class distribution (Outcome):')
print(df['Outcome'].value_counts().sort_index())
print('\nClass percentages:')
print((df['Outcome'].value_counts(normalize=True).sort_index() * 100).round(2))

Class distribution (Outcome):
Outcome
0    500
1    268
Name: count, dtype: int64

Class percentages:
Outcome
0    65.1
1    34.9
Name: proportion, dtype: float64


## Cleaning Pipeline

Recommended for this dataset:
1. Replace impossible zeros with `NaN` in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`.
2. Impute those missing values with median.
3. Optional: clip outliers by IQR bounds.

In [ ]:
clean_df = df.copy()

clean_df[medical_zero_cols] = clean_df[medical_zero_cols].replace(0, np.nan)

median_values = clean_df[medical_zero_cols].median()
clean_df[medical_zero_cols] = clean_df[medical_zero_cols].fillna(median_values)

print('Missing values after median imputation:')
print(clean_df[medical_zero_cols].isna().sum())
clean_df.head()

Missing values after median imputation:
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


In [8]:
def clip_iqr(dataframe, cols):
    capped = dataframe.copy()
    for col in cols:
        q1 = capped[col].quantile(0.25)
        q3 = capped[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        capped[col] = capped[col].clip(lower=lower, upper=upper)
    return capped

numeric_cols = [c for c in clean_df.columns if c != 'Outcome']
clean_df_capped = clip_iqr(clean_df, numeric_cols)

print('Outlier report after optional clipping:')
iqr_outlier_report(clean_df_capped)

Outlier report after optional clipping:


,column,outlier_count,outlier_percent,lower_bound,upper_bound
0,Pregnancies,0,0.0,-6.500,13.500
1,Glucose,0,0.0,39.000,201.000
2,BloodPressure,0,0.0,40.000,104.000
3,SkinThickness,0,0.0,14.500,42.500
4,Insulin,0,0.0,112.875,135.875
5,BMI,0,0.0,13.850,50.250
6,DiabetesPedigreeFunction,0,0.0,-0.330,1.200
7,Age,0,0.0,-1.500,66.500
8,Outcome,0,0.0,-1.500,2.500


In [9]:
# Save cleaned versions
clean_df.to_csv('diabetes_cleaned_median.csv', index=False)
clean_df_capped.to_csv('diabetes_cleaned_median_iqr_capped.csv', index=False)

print('Saved: diabetes_cleaned_median.csv')
print('Saved: diabetes_cleaned_median_iqr_capped.csv')

Saved: diabetes_cleaned_median.csv
Saved: diabetes_cleaned_median_iqr_capped.csv


## Summary

From this dataset, common cleaning needs are:
- No duplicate rows and no explicit `NaN` values in the raw file
- But many impossible zeros in medical columns (especially `Insulin` and `SkinThickness`)
- Noticeable outliers in several numerical features

Use `diabetes_cleaned_median.csv` as a baseline cleaned file, and `diabetes_cleaned_median_iqr_capped.csv` if you want outlier-capped training data.

## Model Benchmarking (Classification)

The target column `Outcome` is binary (0/1), so this is a **classification** problem.
`LogisticRegression` is included because it is a regression-based classifier, and we also compare multiple tree, boosting, distance-based, and probabilistic models.

This section builds a **comparison platform** using:
- Stratified 5-fold cross-validation metrics
- Holdout test metrics
- Composite ranking score across metrics

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

# Use the cleaned dataframe generated above.
# If this cell is run independently, rebuild clean_df.
if 'clean_df' not in globals():
    base_df = pd.read_csv('diabetes.csv').drop_duplicates()
    medical_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    base_df[medical_zero_cols] = base_df[medical_zero_cols].replace(0, np.nan)
    clean_df = base_df.fillna(base_df.mean(numeric_only=True))

X = clean_df.drop(columns=['Outcome'])
y = clean_df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=9))
    ]),
    'SVC-RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=400, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=400, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'GaussianNB': GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

rows = []
for name, model in models.items():
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    rows.append({
        'model': name,
        'cv_accuracy_mean': cv_scores['test_accuracy'].mean(),
        'cv_precision_mean': cv_scores['test_precision'].mean(),
        'cv_recall_mean': cv_scores['test_recall'].mean(),
        'cv_f1_mean': cv_scores['test_f1'].mean(),
        'cv_roc_auc_mean': cv_scores['test_roc_auc'].mean(),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred),
        'test_recall': recall_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred),
        'test_roc_auc': roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan,
    })

comparison_df = pd.DataFrame(rows)

# Composite ranking platform: average rank over key CV metrics.
rank_metrics = ['cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean']
for m in rank_metrics:
    comparison_df[f'rank_{m}'] = comparison_df[m].rank(ascending=False, method='min')
comparison_df['rank_avg'] = comparison_df[[f'rank_{m}' for m in rank_metrics]].mean(axis=1)

leaderboard = comparison_df.sort_values(['cv_roc_auc_mean', 'cv_f1_mean'], ascending=False).reset_index(drop=True)
ranked_platform = comparison_df.sort_values(['rank_avg', 'cv_roc_auc_mean'], ascending=[True, False]).reset_index(drop=True)

print('Leaderboard by CV ROC-AUC then CV F1:')
display(leaderboard[['model', 'cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']].round(4))

print('Composite ranking platform (lower rank_avg is better):')
display(ranked_platform[['model', 'rank_avg', 'cv_accuracy_mean', 'cv_precision_mean', 'cv_recall_mean', 'cv_f1_mean', 'cv_roc_auc_mean']].round(4))

best_cv_auc = leaderboard.loc[0, 'model']
best_composite = ranked_platform.loc[0, 'model']

print(f'Best model by CV ROC-AUC: {best_cv_auc}')
print(f'Best model by composite rank: {best_composite}')